###Stream Customers Data From Cloud Files using autoloader
---------------------------------------------------------------
1.  Read Files from cloud storage using AutoLoader
2. transform the data to add the following columns
    a.file_path: Cloud file path
    b. Ingestion date: Current Timestamp
3. Write the transformed data stream to Delta lake Table

###### - https://docs.databricks.com/aws/en/structured-streaming/concepts

####1. Read using DataStreamReading API

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType

customer_schema = StructType(fields=[StructField("customer_id", IntegerType()),
                                     StructField("customer_name", StringType()),
                                     StructField("date_of_birth", DateType()),
                                     StructField("telephone", StringType()),
                                     StructField("email", StringType()),
                                     StructField("member_since", DateType()),
                                     StructField("created_timesatamp", TimestampType())
                                    ]
                             )

In [0]:
customer_df = (
                spark.readStream
                    .format("json")
                    .schema(customer_schema)
                    .load("/Volumes/gizmobox/landing/operational_data/customers_stream/")
)

####2. Transform the data to add the following columns 
---------------------------------------------------------------
1. file_path: Cloud file path 
2. Ingestion date: Current Timestamp


In [0]:
from pyspark.sql import functions as F
customer_trasformed_df = (
                    customer_df
                    .withColumn("file_path", F.col("_metadata.file_path"))
                    .withColumn("ingestion_date", F.current_timestamp())
)

####3. Write the transformed data stream to Delta lake Table

In [0]:
streaming_query = (
            customer_trasformed_df
            .writeStream.format("delta")
            .trigger(once=True)  ## Other options are "availableNow", "processingTime
            .outputMode("append") ## other options are "update" and "complete"
            .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_stream/_checkpoint_stream")
            .toTable("gizmobox.bronze.customers_stream")
)

In [0]:
streaming_query = (
            customer_trasformed_df
            .writeStream.format("delta")
            .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_stream/_checkpoint_stream")
            .toTable("gizmobox.bronze.customers_stream")
)

In [0]:
streaming_query.stop()

In [0]:
%sql
SELECT * FROM  gizmobox.bronze.customers_stream